# Phase 0 — EDA (step by step)

Exploratory analysis for [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection).

We build this notebook incrementally. **Step 1:** load raw files as they ship from the competition — no column renaming, no rescaling, no derived features.

**Run on Kaggle** for real data. Locally: `python scripts/create_sample_data.py` first.

Confirmed findings go in **[docs/PROJECT_LOG.md](../docs/PROJECT_LOG.md)**.

Sync to Kaggle: `python scripts/sync_kaggle_eda.py --push`

## Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import pydicom

# Resolve repo root when running locally from notebooks/
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src" / "rsna_knee").is_dir():
        REPO_ROOT = candidate
        break
else:
    REPO_ROOT = Path.cwd().parent

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from rsna_knee.constants import (
    SAMPLE_SUBMISSION_CSV,
    TEST_CSV,
    TEST_SERIES_CSV,
    TRAIN_CSV,
    TRAIN_SERIES_CSV,
    TRAIN_SERIES_DIR,
)
from rsna_knee.data.schema import read_competition_csv
from rsna_knee.utils.paths import default_data_root, is_kaggle_kernel

DATA_ROOT = default_data_root()
print(f"Environment : {'Kaggle' if is_kaggle_kernel() else 'local'}")
print(f"Data root   : {DATA_ROOT}")

Environment : local
Data root   : C:\SBX\Kaggle\RSNA_knee_abnormality_detection\data\sample


## Step 1 — Raw CSV tables

Read competition CSVs exactly as stored on disk. Column names and dtypes are unchanged.

In [2]:
csv_files = {
    "train": TRAIN_CSV,
    "train_series": TRAIN_SERIES_CSV,
    "test": TEST_CSV,
    "test_series": TEST_SERIES_CSV,
    "sample_submission": SAMPLE_SUBMISSION_CSV,
}

raw_tables: dict[str, pd.DataFrame] = {}
for name, filename in csv_files.items():
    path = DATA_ROOT / filename
    raw_tables[name] = read_competition_csv(path)
    print(f"{name:18s}  {path.name:22s}  rows={len(raw_tables[name]):,}  cols={len(raw_tables[name].columns)}")

train               train.csv               rows=4  cols=15
train_series        train_series.csv        rows=8  cols=3
test                test.csv                rows=2  cols=2
test_series         test_series.csv         rows=4  cols=3
sample_submission   sample_submission.csv   rows=2  cols=13


In [3]:
for name, df in raw_tables.items():
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    print("Columns:", list(df.columns))
    display(df.head(3))
    df.info()


train
Columns: ['StudyInstanceUID', 'PatientSex', 'Report', 'acl_tear', 'mcl_tear', 'medial_meniscus_injury', 'lateral_meniscus_injury', 'medial_osteoarthritis', 'lateral_osteoarthritis', 'patellofemoral_osteoarthritis', 'joint_effusion', 'synovitis', 'bakers_cyst', 'bone_contusion', 'fracture']


,StudyInstanceUID,PatientSex,Report,acl_tear,mcl_tear,medial_meniscus_injury,lateral_meniscus_injury,medial_osteoarthritis,lateral_osteoarthritis,patellofemoral_osteoarthritis,joint_effusion,synovitis,bakers_cyst,bone_contusion,fracture
0,1.2.826.0.1.3680043.8.498.63688541805381976758...,F,Sample radiology report for local testing.,1,0,1,0,0,1,1,1,0,0,0,1
1,1.2.826.0.1.3680043.8.498.80349276975585802950...,F,Sample radiology report for local testing.,1,0,0,0,0,1,0,1,0,1,1,1
2,1.2.826.0.1.3680043.8.498.43679638807814353751...,F,Sample radiology report for local testing.,0,0,0,0,0,1,1,0,0,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   StudyInstanceUID               4 non-null      str  
 1   PatientSex                     4 non-null      str  
 2   Report                         4 non-null      str  
 3   acl_tear                       4 non-null      int64
 4   mcl_tear                       4 non-null      int64
 5   medial_meniscus_injury         4 non-null      int64
 6   lateral_meniscus_injury        4 non-null      int64
 7   medial_osteoarthritis          4 non-null      int64
 8   lateral_osteoarthritis         4 non-null      int64
 9   patellofemoral_osteoarthritis  4 non-null      int64
 10  joint_effusion                 4 non-null      int64
 11  synovitis                      4 non-null      int64
 12  bakers_cyst                    4 non-null      int64
 13  bone_contusion                 4 no

,StudyInstanceUID,SeriesInstanceUID,fluid_sensitive
0,1.2.826.0.1.3680043.8.498.63688541805381976758...,1.2.826.0.1.3680043.8.498.61206046594374897316...,1
1,1.2.826.0.1.3680043.8.498.63688541805381976758...,1.2.826.0.1.3680043.8.498.17868770850340825900...,0
2,1.2.826.0.1.3680043.8.498.80349276975585802950...,1.2.826.0.1.3680043.8.498.81691595726931730486...,1


<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   StudyInstanceUID   8 non-null      str  
 1   SeriesInstanceUID  8 non-null      str  
 2   fluid_sensitive    8 non-null      int64
dtypes: int64(1), str(2)
memory usage: 324.0 bytes

test
Columns: ['StudyInstanceUID', 'PatientSex']


...

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   StudyInstanceUID  2 non-null      str  
 1   PatientSex        2 non-null      str  
dtypes: str(2)
memory usage: 164.0 bytes

test_series
Columns: ['StudyInstanceUID', 'SeriesInstanceUID', 'fluid_sensitive']


,StudyInstanceUID,SeriesInstanceUID,fluid_sensitive
0,1.2.826.0.1.3680043.8.498.37487830845591055885...,1.2.826.0.1.3680043.8.498.29467186757504221224...,1
1,1.2.826.0.1.3680043.8.498.37487830845591055885...,1.2.826.0.1.3680043.8.498.56057585161364997005...,0
2,1.2.826.0.1.3680043.8.498.26088752430730183477...,1.2.826.0.1.3680043.8.498.91361949157314330571...,1


<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   StudyInstanceUID   4 non-null      str  
 1   SeriesInstanceUID  4 non-null      str  
 2   fluid_sensitive    4 non-null      int64
dtypes: int64(1), str(2)
memory usage: 228.0 bytes

sample_submission
Columns: ['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.37487830845591055885...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,1.2.826.0.1.3680043.8.498.26088752430730183477...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   StudyInstanceUID  2 non-null      str    
 1   ACL               2 non-null      float64
 2   MCL               2 non-null      float64
 3   Medial Meniscus   2 non-null      float64
 4   Lateral Meniscus  2 non-null      float64
 5   Medial OA         2 non-null      float64
 6   Lateral OA        2 non-null      float64
 7   PF OA             2 non-null      float64
 8   Effusion          2 non-null      float64
 9   Synovitis         2 non-null      float64
 10  Baker's           2 non-null      float64
 11  Contusion         2 non-null      float64
 12  Fracture          2 non-null      float64
dtypes: float64(12), str(1)
memory usage: 340.0 bytes


## Step 2 — Raw DICOM files

Peek at one series on disk: file layout, metadata tags, and unstretched pixel values.

In [4]:
series_root = DATA_ROOT / TRAIN_SERIES_DIR
study_dirs = sorted(p for p in series_root.iterdir() if p.is_dir())
print(f"Studies under {series_root}: {len(study_dirs)}")

sample_study = study_dirs[0]
series_dirs = sorted(p for p in sample_study.iterdir() if p.is_dir())
sample_series = series_dirs[0]
dcm_files = sorted(sample_series.glob("*.dcm"))

print(f"Sample study : {sample_study.name}")
print(f"Sample series: {sample_series.name}")
print(f"DICOM files  : {len(dcm_files)} (*.dcm in series folder)")
print("First 5 files:")
for path in dcm_files[:5]:
    print(f"  {path.name}")

Studies under C:\SBX\Kaggle\RSNA_knee_abnormality_detection\data\sample\train_series: 8
Sample study : 1.2.826.0.1.3680043.8.498.25703387297284175225613500901024426257
Sample series: 1.2.826.0.1.3680043.8.498.46553536156281067057238811791810701951
DICOM files  : 8 (*.dcm in series folder)
First 5 files:
  slice_0001.dcm
  slice_0002.dcm
  slice_0003.dcm
  slice_0004.dcm
  slice_0005.dcm


In [5]:
ds = pydicom.dcmread(str(dcm_files[0]))

print("Key DICOM tags (first slice):")
for tag in (
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SOPInstanceUID",
    "InstanceNumber",
    "Modality",
    "Rows",
    "Columns",
    "PixelSpacing",
    "SliceThickness",
    "ImagePositionPatient",
    "RescaleSlope",
    "RescaleIntercept",
    "TransferSyntaxUID",
):
    print(f"  {tag:22s}  {getattr(ds, tag, '<missing>')}")

pixels = ds.pixel_array
print(f"\nRaw pixel_array: shape={pixels.shape}, dtype={pixels.dtype}")
print(f"  min={pixels.min()}, max={pixels.max()}, mean={pixels.mean():.2f}")

Key DICOM tags (first slice):
  StudyInstanceUID        1.2.826.0.1.3680043.8.498.25703387297284175225613500901024426257
  SeriesInstanceUID       1.2.826.0.1.3680043.8.498.46553536156281067057238811791810701951
  SOPInstanceUID          1.2.826.0.1.3680043.8.498.69051518951628281160808319597604075694
  InstanceNumber          1
  Modality                MR
  Rows                    64
  Columns                 64
  PixelSpacing            [0.5, 0.5]
  SliceThickness          3.0
  ImagePositionPatient    [0.0, 0.0, 1.0]
  RescaleSlope            1.0
  RescaleIntercept        0.0
  TransferSyntaxUID       <missing>

Raw pixel_array: shape=(64, 64), dtype=uint16
  min=202, max=824, mean=517.50


### View one slice

Yes — `pixel_array` **is the MRI image** for that slice: a 2D grid of signal intensities.

- One `.dcm` file = **one 2D slice** (shape `Rows × Columns`, e.g. 512×512 on real data).
- All slices in a series folder = **one 3D MRI volume** (stack along the slice axis).
- `Modality: MR` confirms magnetic resonance imaging.
- Values are raw scanner units (often 12–16 bit). We have **not** applied `RescaleSlope` / `RescaleIntercept` yet — that comes later if we want Hounsfield-like or standardized intensities.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Middle slice often shows more anatomy than the first/last
mid_idx = len(dcm_files) // 2
slices_to_show = [0, mid_idx, len(dcm_files) - 1]

for ax, idx in zip(axes, slices_to_show):
    ds_i = pydicom.dcmread(str(dcm_files[idx]))
    img = ds_i.pixel_array
    ax.imshow(img, cmap="gray")
    ax.set_title(f"slice {idx + 1} / {len(dcm_files)}")
    ax.axis("off")

plt.suptitle(f"Raw pixel_array — series {sample_series.name[:20]}…")
plt.tight_layout()
plt.show()